# IFCE Campus Tauá -- Tecnologia em Análise e Desenvolvimento de Sistemas (ADS)
## Disciplina: Bancos de Dados Não-Relacionais (ADS16)
### Aula 01: O Paradigma NoSQL, Descasamento de Impedância & Primeiro Banco MongoDB
**Professor:** Me. Reginaldo Fernandes | **Semestre:** 2026.2

---

### 📚 Referências Bibliográficas Integradas:
1. **SADALAGE, Pramod J.; FOWLER, Martin.** *NoSQL Essencial: Um Guia Conciso para o Mundo Emergente da Persistência Poliglota.* São Paulo: Novatec Editora, 2014.
2. **FROZZA, Angelo Augusto; SCHREINER, Geomar André; MELLO, Ronaldo dos Santos.** *Projeto de Bancos de Dados NoSQL.* In: Minicurso SBC, Capítulo 2.
3. **SHAPIRA, Gwen; PALINO, Todd; SIVARAM, Rajini; PETTY, Krit.** *Kafka: The Definitive Guide - Real-Time Data and Stream Processing at Scale.* 2. ed. O'Reilly Media, 2021.
4. **ELMASRI, Ramez; NAVATHE, Shamkant B.** *Sistemas de Banco de Dados.* 7. ed. São Paulo: Pearson Education do Brasil, 2018.
5. **PANIZ, David.** *NoSQL: como armazenar os dados de uma aplicação moderna.* São Paulo: Casa do Código, 2016.

---

### 🎯 Objetivos Deste Caderno Prático:
1. Analisar na prática o fenômeno do **Descasamento de Impedância Objeto-Relacional** (*Object-Relational Impedance Mismatch*).
2. Comparar a complexidade e latência de operações em tabelas relacionais normalizadas (3NF) com a simplicidade atômica do modelo de documentos agregados (JSON/BSON).
3. Aprender a instalar e gerenciar o **Serviço do MongoDB** no **Ubuntu (v8.0/v9.0)** e no **Windows**.
4. Executar operações práticas de criação de banco, inserção agregada, busca aninhada e atualização atômica no MongoDB via código Python e terminal `mongosh`.


### 1. Contexto Teórico: O Ponto de Inflexão dos Dados

Por mais de três décadas (desde Edgar F. Codd em 1970), o modelo relacional reinou absoluto. No entanto, o advento da Web 2.0 e do Big Data impôs três grandes limitações aos RDBMS centralizados:

1. **Escala Vertical (*Scale-Up*) vs Escala Horizontal (*Scale-Out*):**
   - *Scale-Up:* Adquirir servidores com mais CPUs e mais RAM gera custos exponenciais ($$$$) e atinge um teto físico. Além disso, cria um **Ponto Único de Falha (SPOF)**.
   - *Scale-Out:* O modelo NoSQL distribui dados em clusters de nós convencionais (*commodity hardware*), com custo linear e alta resiliência via replicação.
2. **O Custo dos JOINs e o *Network Shuffle*:**
   - Em uma única máquina, um `JOIN` acessa memória RAM ou disco local em fração de milissegundo.
   - Em clusters particionados, cruzar dados exige tráfego massivo pela rede física (*Network Shuffle*), aumentando a latência de milissegundos para dezenas de segundos!
3. **Orientação a Agregados (*Aggregate Orientation* - Sadalage & Fowler, 2014):**
   - Chave-Valor, Documentos e Família de Colunas agrupam dados relacionados em uma **unidade atômica de acesso indivisível** (o Agregado).
   - O modelo de Grafos e o Relacional são **não-orientados a agregados**, priorizando interconexões complexas.


### 2. Definição do Domínio de Negócio em Python (OOP)
Abaixo, modelamos uma entidade de e-commerce típica utilizando `dataclasses` em Python. Observe como o objeto `Cliente` naturalmente possui um objeto aninhado `Endereco` e uma lista de objetos `Pedido`, cada qual com seus respectivos `ItemPedido`.


In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Any
import json
import sqlite3
import time
import socket

@dataclass
class ItemPedido:
    produto: str
    quantidade: int
    preco_unitario: float

@dataclass
class Pedido:
    id_pedido: str
    data_emissao: str
    itens: List[ItemPedido] = field(default_factory=list)

    @property
    def total(self) -> float:
        return sum(item.quantidade * item.preco_unitario for item in self.itens)

@dataclass
class Cliente:
    id_cliente: int
    nome: str
    email: str
    cidade: str
    uf: str
    pedidos: List[Pedido] = field(default_factory=list)

# Instanciando dados de teste do IFCE Campus Tauá
cliente_exemplo = Cliente(
    id_cliente=1,
    nome="Maria Clara Silva",
    email="maria.clara@aluno.ifce.edu.br",
    cidade="Tauá",
    uf="CE",
    pedidos=[
        Pedido(
            id_pedido="PED-2026-001",
            data_emissao="2026-09-29",
            itens=[
                ItemPedido("Notebook Dell", 1, 4500.00),
                ItemPedido("Mouse Sem Fio", 2, 85.50)
            ]
        ),
        Pedido(
            id_pedido="PED-2026-002",
            data_emissao="2026-10-01",
            itens=[ItemPedido("Monitor 27 Pol", 1, 1200.00)]
        )
    ]
)

print(f"✅ Objeto em Memória Carregado:")
print(f"   Cliente: {cliente_exemplo.nome} ({cliente_exemplo.cidade}/{cliente_exemplo.uf})")
print(f"   Total de Pedidos: {len(cliente_exemplo.pedidos)}")


### 3. Abordagem Relacional: Fragmentação Normalizada (3NF) & O Atrito do ORM
No paradigma relacional, uma entidade rica em objetos precisa ser fatiada em 4 tabelas separadas ligadas por chaves estrangeiras:
`clientes`, `enderecos`, `pedidos` e `itens_pedido`.


In [ ]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Criação das 4 tabelas normalizadas
cursor.executescript('''
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY,
        nome TEXT NOT NULL,
        email TEXT NOT NULL
    );
    CREATE TABLE enderecos (
        id_cliente INTEGER PRIMARY KEY,
        cidade TEXT NOT NULL,
        uf TEXT NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES clientes(id)
    );
    CREATE TABLE pedidos (
        id_pedido TEXT PRIMARY KEY,
        id_cliente INTEGER NOT NULL,
        data_emissao TEXT NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES clientes(id)
    );
    CREATE TABLE itens_pedido (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        id_pedido TEXT NOT NULL,
        produto TEXT NOT NULL,
        quantidade INTEGER NOT NULL,
        preco_unitario REAL NOT NULL,
        FOREIGN KEY (id_pedido) REFERENCES pedidos(id_pedido)
    );
''')

t0 = time.perf_counter()

# Inserção fragmentada obrigatória (4 tabelas)
cursor.execute("INSERT INTO clientes VALUES (?, ?, ?)", 
               (cliente_exemplo.id_cliente, cliente_exemplo.nome, cliente_exemplo.email))
cursor.execute("INSERT INTO enderecos VALUES (?, ?, ?)", 
               (cliente_exemplo.id_cliente, cliente_exemplo.cidade, cliente_exemplo.uf))

for pedido in cliente_exemplo.pedidos:
    cursor.execute("INSERT INTO pedidos VALUES (?, ?, ?)", 
                   (pedido.id_pedido, cliente_exemplo.id_cliente, pedido.data_emissao))
    for item in pedido.itens:
        cursor.execute("INSERT INTO itens_pedido (id_pedido, produto, quantidade, preco_unitario) VALUES (?, ?, ?, ?)",
                       (pedido.id_pedido, item.produto, item.quantidade, item.preco_unitario))
conn.commit()
t_rel_insert = (time.perf_counter() - t0) * 1000

print(f"⏱️  Tempo de escrita relacional (4 tabelas): {t_rel_insert:.4f} ms")


#### Leitura Relacional: A Necessidade de Remontagem via Múltiplos JOINs
Ao consultar os dados, o SQL retorna uma tabela bidimensional com forte redundância nos dados do cliente e pedidos a cada linha de item:


In [ ]:
sql_consulta = '''
    SELECT c.nome, e.cidade, e.uf, p.id_pedido, p.data_emissao, i.produto, i.quantidade, i.preco_unitario
    FROM clientes c
    JOIN enderecos e ON c.id = e.id_cliente
    JOIN pedidos p ON c.id = p.id_cliente
    JOIN itens_pedido i ON p.id_pedido = i.id_pedido
    WHERE c.id = 1;
'''
cursor.execute(sql_consulta)
linhas = cursor.fetchall()

print(f"Linhas retornadas pela junção:")
for linha in linhas:
    print(f"  -> {linha}")

print("
⚠️  Atrito Objeto-Relacional:")
print("   A aplicação precisa iterar manualmente pelas tuplas para agrupar e reconstruir o objeto em memória.")


### 4. Abordagem Não-Relacional: Orientação a Agregados em Documentos (JSON/BSON)
Em bancos NoSQL orientados a documentos como o **MongoDB**, a entidade completa é persistida como um agregado hierárquico em um único registro atômico:


In [ ]:
t0 = time.perf_counter()

documento_nosql = {
    "_id": cliente_exemplo.id_cliente,
    "nome": cliente_exemplo.nome,
    "email": cliente_exemplo.email,
    "endereco": {
        "cidade": cliente_exemplo.cidade,
        "uf": cliente_exemplo.uf
    },
    "pedidos": [
        {
            "id_pedido": p.id_pedido,
            "data_emissao": p.data_emissao,
            "total": p.total,
            "itens": [
                {
                    "produto": item.produto,
                    "quantidade": item.quantidade,
                    "preco_unitario": item.preco_unitario
                } for item in p.itens
            ]
        } for p in cliente_exemplo.pedidos
    ]
}

doc_json = json.dumps(documento_nosql, indent=2, ensure_ascii=False)
t_nosql_serial = (time.perf_counter() - t0) * 1000

print("✅ Documento NoSQL Agregado:")
print(doc_json)
print(f"
⏱️  Tempo de serialização: {t_nosql_serial:.4f} ms")
print("💡 Vantagem: Leitura e escrita direta em 1 único I/O de disco sem nenhum JOIN!")


### 5. Guia Oficial: Como Instalar o Serviço MongoDB

O MongoDB opera como um serviço em segundo plano (*daemon* `mongod`) escutando por padrão na porta `27017`.

#### A) Instalação no Ubuntu (v8.0 / v9.0 LTS)
```bash
# 1. Instalar dependências prévias
sudo apt update && sudo apt install -y gnupg curl

# 2. Importar a chave pública GPG oficial
curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc | \
  sudo gpg -o /usr/share/keyrings/mongodb-server-8.0.gpg --dearmor

# 3. Adicionar o repositório oficial da distribuição
echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] \
  https://repo.mongodb.org/apt/ubuntu $(lsb_release -cs)/mongodb-org/8.0 multiverse" | \
  sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list

# 4. Atualizar o APT e instalar o pacote oficial
sudo apt update && sudo apt install -y mongodb-org

# 5. Gerenciar o serviço com systemd
sudo systemctl start mongod      # Inicia o daemon
sudo systemctl status mongod     # Confere status ativo
sudo systemctl enable mongod     # Ativa na inicialização
mongosh                          # Abre o shell interativo
```

#### B) Instalação no Windows (Community Server)
1. Acesse o [MongoDB Download Center](https://www.mongodb.com/try/download/community) e baixe o instalador `.msi` mais recente.
2. No assistente, escolha a opção **"Complete"**.
3. Na tela de configuração de serviço, mantenha marcada a caixa:
   - `[X] Install MongoDB as a Service` (Nome: `MongoDB`).
4. Recomenda-se manter marcado o **MongoDB Compass** (interface gráfica oficial).
5. Gerenciamento do Serviço no Windows:
   - **PowerShell (Administrador):**
     ```powershell
     Get-Service MongoDB        # Status
     Start-Service MongoDB      # Inicia
     Stop-Service MongoDB       # Para
     ```
   - **Prompt CMD (Administrador):**
     ```cmd
     net start MongoDB
     net stop MongoDB
     ```
   - **Painel Visual:** Pressione `Win + R`, digite `services.msc` e localize `MongoDB Server`.
6. Adicione `C:\Program Files\MongoDB\Server\<versao>\bin` ao PATH do sistema e teste com `mongosh`.


### 6. Execução Prática de um Banco no MongoDB

Abaixo, executamos operações em um banco de dados MongoDB local (se o serviço estiver ativo) ou por simulação precisa do motor de documentos:

1. Criação/Seleção do banco de dados `ads_comercio`.
2. Inserção do documento na coleção `clientes`.
3. Consulta por atributo aninhado via *Dot Notation* (`endereco.cidade`).
4. Atualização atômica adicionando um novo item ao pedido via operador `$push` sem realizar nenhum `JOIN`.


In [ ]:
def testar_conexao_mongo(host="localhost", port=27017, timeout=0.5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except (OSError, ConnectionRefusedError):
        return False

mongo_online = testar_conexao_mongo()

if mongo_online:
    try:
        import pymongo
        client = pymongo.MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=2000)
        db = client["ads_comercio"]
        col = db["clientes"]
        
        # Remove teste anterior e insere o agregado
        col.delete_one({"_id": documento_nosql["_id"]})
        col.insert_one(documento_nosql)
        print("🎉 [MongoDB REAL] Documento inserido com sucesso na coleção 'clientes'!")
        
        # Consulta por campo aninhado
        resultado = col.find_one({"endereco.cidade": "Tauá"})
        print(f"🔍 [MongoDB REAL] Consulta por 'endereco.cidade == Tauá':")
        print(f"   Nome: {resultado['nome']} | E-mail: {resultado['email']}")
        
        # Atualização atômica com $push
        col.update_one(
            {"_id": 1},
            {"$push": {"pedidos.0.itens": {"produto": "Teclado Mecânico", "quantidade": 1, "preco_unitario": 250.0}}}
        )
        print("⚡ [MongoDB REAL] Operador $push executado com sucesso!")
        
        doc_atualizado = col.find_one({"_id": 1})
        itens_ped1 = doc_atualizado["pedidos"][0]["itens"]
        print(f"📦 [MongoDB REAL] Total de itens no Pedido 1: {len(itens_ped1)}")
        client.close()
    except Exception as e:
        print(f"⚠️  PyMongo reportou: {e}")
else:
    print("ℹ️  Serviço MongoDB local não detectado na porta 27017.")
    print("   -> No Ubuntu : sudo systemctl start mongod")
    print("   -> No Windows: net start MongoDB")
    print("   -> No Docker : docker run -d -p 27017:27017 mongo:latest")
    print("
🧪 Executando Simulação Didática do Motor de Documentos MongoDB:")
    
    colecao_simulada = {documento_nosql["_id"]: documento_nosql}
    print(f"   insertOne -> Documento inserido com _id: {documento_nosql['_id']}")
    
    # Consulta por dot notation
    encontrados = [d for d in colecao_simulada.values() if d.get("endereco", {}).get("cidade") == "Tauá"]
    print(f'   find({{"endereco.cidade": "Tauá"}}) -> {len(encontrados)} documento retornado.')
    
    # Atualização atômica $push
    colecao_simulada[1]["pedidos"][0]["itens"].append({
        "produto": "Teclado Mecânico", "quantidade": 1, "preco_unitario": 250.0
    })
    print("   updateOne({$push}) -> Novo item adicionado diretamente no array do pedido!")
    print(f"   Total de itens no Pedido 1 após atualização: {len(colecao_simulada[1]['pedidos'][0]['itens'])}")


### 7. Resumo dos Comandos Interativos no `mongosh`

Para executar os mesmos passos diretamente no terminal de comandos do MongoDB (`mongosh`):

```javascript
// 1. Alternar para o banco de dados da aula
use ads_comercio;

// 2. Inserir documento agregado
db.clientes.insertOne({
  _id: 1,
  nome: "Maria Clara Silva",
  email: "maria.clara@aluno.ifce.edu.br",
  endereco: { cidade: "Tauá", uf: "CE" },
  pedidos: [
    {
      id_pedido: "PED-2026-001",
      data_emissao: "2026-09-29",
      itens: [
        { produto: "Notebook Dell", quantidade: 1, preco_unitario: 4500.00 },
        { produto: "Mouse Sem Fio", quantidade: 2, preco_unitario: 85.50 }
      ],
      total: 4671.00
    }
  ]
});

// 3. Consultar cliente filtrando por propriedade aninhada
db.clientes.find({ "endereco.cidade": "Tauá" }).pretty();

// 4. Atualizar atomicamente adicionando um produto ao pedido sem JOINs
db.clientes.updateOne(
  { _id: 1 },
  { $push: { "pedidos.0.itens": { produto: "Teclado Mecânico", quantidade: 1, preco_unitario: 250.00 } } }
);

// 5. Verificar a contagem de documentos
db.clientes.countDocuments();
```

---
**Parabéns!** Você concluiu a Aula 01 e aprendeu os fundamentos da persistência poliglota, orientação a agregados e como criar seu primeiro banco de dados MongoDB!
